In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l2.4 Embeddings

A token id is just a number; the model needs a vector. An embedding table is a
lookup: row `i` is the learned vector for token `i`. Training shapes those rows.

In [ ]:
from torch import nn

vocab, dim = 50, 8
emb = nn.Embedding(vocab, dim)
ids = torch.tensor([1, 2, 3])
vecs = emb(ids)
print("ids", ids.tolist(), "-> vectors of shape", tuple(vecs.shape))

The same id always maps to the same row, so identical tokens start identical
and only diverge through context in later layers.

In [ ]:
assert tuple(vecs.shape) == (3, 8)
assert torch.equal(emb(torch.tensor([1]))[0], emb.weight[1])